# HolmesVAU sampler benchmark on UCF-Crime and XD-Violence
Runtime → T4 GPU. Run top to bottom. Outputs are written to Google Drive so a disconnect never loses work; every stage resumes where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/HolmesVAU && git clone -q -b thesis-benchmark https://github.com/topcoder1804/HolmesVAU-Thesis.git /content/HolmesVAU
%cd /content/HolmesVAU
!mkdir -p /content/drive/MyDrive/HolmesVAU/benchmark

## Environment (Python 3.10, same pins as the earlier Fast-ATS runs)

In [ ]:
!pip install -q uv
!uv python install 3.10 -q
!uv venv -q --python 3.10 --seed /content/holmes-env
!uv pip install -q --python /content/holmes-env/bin/python 'setuptools==68.2.2' 'packaging==23.2' matplotlib pillow pandas
!uv pip install -q --python /content/holmes-env/bin/python -r requirements.txt
!/content/holmes-env/bin/python -c "import torch;print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
!/content/holmes-env/bin/python -c "from huggingface_hub import snapshot_download as s; s('ppxin321/HolmesVAU-2B', local_dir='/content/HolmesVAU/ckpts/HolmesVAU-2B')"
!du -sh ckpts/HolmesVAU-2B

## 1. UCF-Crime test set (290 videos) → dense CLS features

In [ ]:
!/content/holmes-env/bin/python experiments/benchmark/prepare_data.py --root /content/data --dataset ucf
!cp /content/data/ucf/test_annotations.txt /content/drive/MyDrive/HolmesVAU/benchmark/ucf_test_annotations.txt
!df -h /content | tail -1

In [ ]:
!PYTHONPATH=. /content/holmes-env/bin/python experiments/benchmark/extract_features.py --video-dir /content/data/ucf/videos --out /content/drive/MyDrive/HolmesVAU/benchmark/features/ucf --stride 16 --batch 32

## 2. XD-Violence test set (800 videos) → dense CLS features

In [ ]:
!/content/holmes-env/bin/python experiments/benchmark/prepare_data.py --root /content/data --dataset xd
!cp /content/data/xd/test_annotations.txt /content/drive/MyDrive/HolmesVAU/benchmark/xd_test_annotations.txt

In [ ]:
!PYTHONPATH=. /content/holmes-env/bin/python experiments/benchmark/extract_features.py --video-dir /content/data/xd/videos --out /content/drive/MyDrive/HolmesVAU/benchmark/features/xd --stride 16 --batch 32

## 3. Explanations with HolmesVAU-2B
`selections_*.json` are produced offline by `replay.py` and copied into the Drive folder.

In [ ]:
!PYTHONPATH=. /content/holmes-env/bin/python experiments/benchmark/generate_explanations.py --selections /content/drive/MyDrive/HolmesVAU/benchmark/selections_ucf.json /content/drive/MyDrive/HolmesVAU/benchmark/selections_xd.json --video-dirs /content/data/ucf/videos /content/data/xd/videos --out /content/drive/MyDrive/HolmesVAU/benchmark/explanations.jsonl